Combining all datasets into one large combined datafile

In [16]:
# Combine Shape2SAS Datasets Into One NPZ File

from pathlib import Path
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================

DATASET_DIR = Path.cwd()

OUTPUT_FILE = (
    DATASET_DIR
    / "saxs_training_data_combined.npz"
)

# ============================================================
# DATASETS TO COMBINE
# ============================================================

npz_files = [
    "saxs_training_data_sphere.npz",
    "saxs_training_data_hollow_sphere.npz",
    "saxs_training_data_hollow_sphere2.npz",
    "saxs_training_data_ellipsoid.npz",
    "saxs_training_data_ellipsoid1.npz",
    "saxs_training_data_ellipsoid2.npz",
    "saxs_training_data_ellipsoid3.npz",
    "saxs_training_data_cylinder.npz",
    "saxs_training_data_cylinder1.npz",
    "saxs_training_data_cylinder2.npz",
    "saxs_training_data_elliptical_cylinder.npz",
    "saxs_training_data_elliptical_cylinder1.npz",
    "saxs_training_data_elliptical_cylinder2.npz",
    "saxs_training_data_elliptical_cylinder3.npz",
    "saxs_training_data_cube.npz",
    "saxs_training_data_hollow_cube.npz",
    "saxs_training_data_hollow_cube1.npz",
    "saxs_training_data_hollow_cube2.npz",
    "saxs_training_data_cuboid.npz",
    "saxs_training_data_cuboid1.npz",
    "saxs_training_data_cuboid2.npz",
    "saxs_training_data_cuboid3.npz",
    "saxs_training_data_cuboid4.npz",
    "saxs_training_data_ring.npz",
    "saxs_training_data_ring1.npz",
    "saxs_training_data_ring2.npz",
    "saxs_training_data_ring3.npz",
    "saxs_training_data_hyperboloid.npz",
    "saxs_training_data_hyperboloid1.npz",
    "saxs_training_data_hyperboloid2.npz",
    "saxs_training_data_hyperboloid3.npz",
    "saxs_training_data_hyperboloid4.npz",
    "saxs_training_data_superellipsoid.npz",
    "saxs_training_data_superellipsoid1.npz",
    "saxs_training_data_superellipsoid2.npz",
    "saxs_training_data_superellipsoid3.npz",
    "saxs_training_data_superellipsoid4.npz",
    "saxs_training_data_torus.npz",
    "saxs_training_data_torus1.npz",
    "saxs_training_data_torus2.npz",
]

# ============================================================
# STORAGE
# ============================================================

shape_labels = []

model_labels_list = []

q_list = []

Iq_list = []

Isim_list = []

sigma_list = []

r_list = []

pr_list = []

dmax_list = []

r_grid_list = []

# optional parameter storage
parameter_dict = {}

# ============================================================
# LOAD DATASETS
# ============================================================

for file_name in npz_files:

    path = DATASET_DIR / file_name

    if not path.exists():

        print(
            f"Skipping missing file: {path}"
        )

        continue

    print(
        f"Loading: {path.name}"
    )

    data = np.load(
        path,
        allow_pickle=True
    )

    # --------------------------------------------------------
    # Infer shape name
    # --------------------------------------------------------

    shape_name = (
        path.stem
        .replace(
            "saxs_training_data_",
            ""
        )
    )

    # --------------------------------------------------------
    # Required arrays
    # --------------------------------------------------------

    q_matrix = data["q_matrix"]

    Iq_matrix = data["Iq_matrix"]

    Isim_matrix = data["Isim_matrix"]

    sigma_matrix = data["sigma_matrix"]

    r_matrix = data["r_matrix"]

    pr_matrix = data["pr_matrix"]

    dmax_array = data["dmax_array"]

    r_grid_matrix = data["r_grid_matrix"]
    
    model_labels = data["model_labels"]

    n_samples = Iq_matrix.shape[0]

    # --------------------------------------------------------
    # Store primary data
    # --------------------------------------------------------

    q_list.append(q_matrix)

    Iq_list.append(Iq_matrix)

    Isim_list.append(Isim_matrix)

    sigma_list.append(sigma_matrix)

    r_list.append(r_matrix)

    pr_list.append(pr_matrix)

    dmax_list.append(dmax_array)

    r_grid_list.append(r_grid_matrix)

    # --------------------------------------------------------
    # Shape labels
    # --------------------------------------------------------

    shape_labels.extend(
        [shape_name] * n_samples
    )
    
    model_labels_list.append(
        model_labels
    )

    # --------------------------------------------------------
    # Store parameter arrays automatically
    # --------------------------------------------------------

    for key in data.files:

        if key in {
            "q_matrix",
            "Iq_matrix",
            "Isim_matrix",
            "sigma_matrix",
            "r_matrix",
            "pr_matrix",
            "dmax_array",
            "r_grid_matrix",
        }:
            continue

        arr = data[key]

        if isinstance(
            arr,
            np.ndarray
        ):

            if (
                len(arr.shape) > 0
                and arr.shape[0] == n_samples
            ):

                full_key = (
                    f"{shape_name}_{key}"
                )

                if (
                    full_key
                    not in parameter_dict
                ):
                    parameter_dict[
                        full_key
                    ] = []

                parameter_dict[
                    full_key
                ].append(arr)

# ============================================================
# CONCATENATE DATA
# ============================================================

q_matrix = np.vstack(
    q_list
)

Iq_matrix = np.vstack(
    Iq_list
)

Isim_matrix = np.vstack(
    Isim_list
)

sigma_matrix = np.vstack(
    sigma_list
)

r_matrix = np.vstack(
    r_list
)

pr_matrix = np.vstack(
    pr_list
)

try:

    dmax_array = np.concatenate(
        dmax_list
    )

except Exception:

    dmax_array = np.asarray(
        dmax_list
    )

r_grid_matrix = np.vstack(
    r_grid_list
)

shape_labels = np.asarray(
    shape_labels
)


model_labels = np.concatenate(
    model_labels_list
)

# ============================================================
# CONCATENATE PARAMETER ARRAYS
# ============================================================

combined_parameters = {}

for key, value_list in parameter_dict.items():

    try:

        combined_parameters[key] = (
            np.concatenate(
                value_list
            )
        )

    except Exception:

        combined_parameters[key] = (
            np.asarray(
                value_list,
                dtype=object
            )
        )

# ============================================================
# SUMMARY
# ============================================================

print()
print("FINAL DATASET")
print("-------------")

print(
    "q_matrix:",
    q_matrix.shape
)

print(
    "Iq_matrix:",
    Iq_matrix.shape
)

print(
    "Isim_matrix:",
    Isim_matrix.shape
)

print(
    "sigma_matrix:",
    sigma_matrix.shape
)

print(
    "r_matrix:",
    r_matrix.shape
)

print(
    "pr_matrix:",
    pr_matrix.shape
)

print(
    "dmax_array:",
    dmax_array.shape
)

print(
    "r_grid_matrix:",
    r_grid_matrix.shape
)

print(
    "shape_labels:",
    shape_labels.shape
)

print(
    "model_labels:",
    model_labels.shape
)


# ============================================================
# SAVE COMBINED DATASET
# ============================================================

np.savez_compressed(
    OUTPUT_FILE,

    shape_labels=shape_labels,
    
    model_labels=model_labels,

    q_matrix=q_matrix,

    Iq_matrix=Iq_matrix,

    Isim_matrix=Isim_matrix,

    sigma_matrix=sigma_matrix,

    r_matrix=r_matrix,

    pr_matrix=pr_matrix,

    dmax_array=dmax_array,

    r_grid_matrix=r_grid_matrix,

    **combined_parameters,
)

# ============================================================
# DONE
# ============================================================

print()
print(
    f"Saved combined dataset to:\n"
    f"{OUTPUT_FILE}"
)

Loading: saxs_training_data_sphere.npz
Loading: saxs_training_data_hollow_sphere.npz
Loading: saxs_training_data_hollow_sphere2.npz
Loading: saxs_training_data_ellipsoid.npz
Loading: saxs_training_data_ellipsoid1.npz
Loading: saxs_training_data_ellipsoid2.npz
Loading: saxs_training_data_ellipsoid3.npz
Loading: saxs_training_data_cylinder.npz
Loading: saxs_training_data_cylinder1.npz
Loading: saxs_training_data_cylinder2.npz
Loading: saxs_training_data_elliptical_cylinder.npz
Loading: saxs_training_data_elliptical_cylinder1.npz
Loading: saxs_training_data_elliptical_cylinder2.npz
Loading: saxs_training_data_elliptical_cylinder3.npz
Loading: saxs_training_data_cube.npz
Loading: saxs_training_data_hollow_cube.npz
Loading: saxs_training_data_hollow_cube1.npz
Loading: saxs_training_data_hollow_cube2.npz
Loading: saxs_training_data_cuboid.npz
Loading: saxs_training_data_cuboid1.npz
Loading: saxs_training_data_cuboid2.npz
Loading: saxs_training_data_cuboid3.npz
Loading: saxs_training_data_cub

In [9]:
from pathlib import Path
import numpy as np

DATASET_DIR = Path.cwd()

npz_files = [
    # your file list here
]

all_ok = True

reference_q = None
reference_r = None

for file_name in npz_files:

    path = DATASET_DIR / file_name

    print(f"\nChecking: {file_name}")

    data = np.load(path, allow_pickle=True)

    required = [
        "q_matrix",
        "Iq_matrix",
        "r_matrix",
        "pr_matrix",
        "dmax_array",
        "r_grid_matrix",
    ]

    # -------------------------
    # Required keys
    # -------------------------
    for key in required:
        if key not in data:
            print(f"  Missing key: {key}")
            all_ok = False

    # -------------------------
    # Shape checks
    # -------------------------
    n = data["Iq_matrix"].shape[0]

    for key in required:

        arr = data[key]

        if key == "dmax_array":

            if arr.shape != (n,):
                print(f"  Bad shape for {key}: {arr.shape}")

        else:

            if arr.shape != (n, 512):
                print(f"  Bad shape for {key}: {arr.shape}")

    # -------------------------
    # q-grid consistency
    # -------------------------
    q0 = data["q_matrix"][0]
    r0 = data["r_grid_matrix"][0]

    if reference_q is None:
        reference_q = q0
        reference_r = r0
    else:

        if not np.allclose(q0, reference_q):
            print("  WARNING: q-grid differs")

        if not np.allclose(r0, reference_r):
            print("  WARNING: r-grid differs")

print("\n====================")

if all_ok:
    print("Core dataset structure is VALID")
else:
    print("Dataset structure problems found")

print("====================")


Core dataset structure is VALID


Combining superellipsoid datasets into one dataset

In [18]:
# Combine Shape2SAS Datasets Into One NPZ File

from pathlib import Path
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================

DATASET_DIR = Path.cwd()

OUTPUT_FILE = (
    DATASET_DIR
    / "saxs_training_data_combined_superellipsoid.npz"
)

# ============================================================
# DATASETS TO COMBINE
# ============================================================

npz_files = [
    "saxs_training_data_superellipsoid.npz",
    "saxs_training_data_superellipsoid1.npz",
    "saxs_training_data_superellipsoid2.npz",
    "saxs_training_data_superellipsoid3.npz",
    "saxs_training_data_superellipsoid4.npz",
]

# ============================================================
# STORAGE
# ============================================================

shape_labels = []

model_labels_list = []

q_list = []

Iq_list = []

Isim_list = []

sigma_list = []

r_list = []

pr_list = []

dmax_list = []

r_grid_list = []

# optional parameter storage
parameter_dict = {}

# ============================================================
# LOAD DATASETS
# ============================================================

for file_name in npz_files:

    path = DATASET_DIR / file_name

    if not path.exists():

        print(
            f"Skipping missing file: {path}"
        )

        continue

    print(
        f"Loading: {path.name}"
    )

    data = np.load(
        path,
        allow_pickle=True
    )

    # --------------------------------------------------------
    # Infer shape name
    # --------------------------------------------------------

    shape_name = (
        path.stem
        .replace(
            "saxs_training_data_",
            ""
        )
    )

    # --------------------------------------------------------
    # Required arrays
    # --------------------------------------------------------

    q_matrix = data["q_matrix"]

    Iq_matrix = data["Iq_matrix"]

    Isim_matrix = data["Isim_matrix"]

    sigma_matrix = data["sigma_matrix"]

    r_matrix = data["r_matrix"]

    pr_matrix = data["pr_matrix"]

    dmax_array = data["dmax_array"]

    r_grid_matrix = data["r_grid_matrix"]
    
    model_labels = data["model_labels"]

    n_samples = Iq_matrix.shape[0]

    # --------------------------------------------------------
    # Store primary data
    # --------------------------------------------------------

    q_list.append(q_matrix)

    Iq_list.append(Iq_matrix)

    Isim_list.append(Isim_matrix)

    sigma_list.append(sigma_matrix)

    r_list.append(r_matrix)

    pr_list.append(pr_matrix)

    dmax_list.append(dmax_array)

    r_grid_list.append(r_grid_matrix)

    # --------------------------------------------------------
    # Shape labels
    # --------------------------------------------------------

    shape_labels.extend(
        [shape_name] * n_samples
    )
    
    model_labels_list.append(
        model_labels
    )

    # --------------------------------------------------------
    # Store parameter arrays automatically
    # --------------------------------------------------------

    for key in data.files:

        if key in {
            "q_matrix",
            "Iq_matrix",
            "Isim_matrix",
            "sigma_matrix",
            "r_matrix",
            "pr_matrix",
            "dmax_array",
            "r_grid_matrix",
        }:
            continue

        arr = data[key]

        if isinstance(
            arr,
            np.ndarray
        ):

            if (
                len(arr.shape) > 0
                and arr.shape[0] == n_samples
            ):

                full_key = (
                    f"{shape_name}_{key}"
                )

                if (
                    full_key
                    not in parameter_dict
                ):
                    parameter_dict[
                        full_key
                    ] = []

                parameter_dict[
                    full_key
                ].append(arr)

# ============================================================
# CONCATENATE DATA
# ============================================================

q_matrix = np.vstack(
    q_list
)

Iq_matrix = np.vstack(
    Iq_list
)

Isim_matrix = np.vstack(
    Isim_list
)

sigma_matrix = np.vstack(
    sigma_list
)

r_matrix = np.vstack(
    r_list
)

pr_matrix = np.vstack(
    pr_list
)

try:

    dmax_array = np.concatenate(
        dmax_list
    )

except Exception:

    dmax_array = np.asarray(
        dmax_list
    )

r_grid_matrix = np.vstack(
    r_grid_list
)

shape_labels = np.asarray(
    shape_labels
)


model_labels = np.concatenate(
    model_labels_list
)

# ============================================================
# CONCATENATE PARAMETER ARRAYS
# ============================================================

combined_parameters = {}

for key, value_list in parameter_dict.items():

    try:

        combined_parameters[key] = (
            np.concatenate(
                value_list
            )
        )

    except Exception:

        combined_parameters[key] = (
            np.asarray(
                value_list,
                dtype=object
            )
        )

# ============================================================
# SUMMARY
# ============================================================

print()
print("FINAL DATASET")
print("-------------")

print(
    "q_matrix:",
    q_matrix.shape
)

print(
    "Iq_matrix:",
    Iq_matrix.shape
)

print(
    "Isim_matrix:",
    Isim_matrix.shape
)

print(
    "sigma_matrix:",
    sigma_matrix.shape
)

print(
    "r_matrix:",
    r_matrix.shape
)

print(
    "pr_matrix:",
    pr_matrix.shape
)

print(
    "dmax_array:",
    dmax_array.shape
)

print(
    "r_grid_matrix:",
    r_grid_matrix.shape
)

print(
    "shape_labels:",
    shape_labels.shape
)

print(
    "model_labels:",
    model_labels.shape
)


# ============================================================
# SAVE COMBINED DATASET
# ============================================================

np.savez_compressed(
    OUTPUT_FILE,

    shape_labels=shape_labels,
    
    model_labels=model_labels,

    q_matrix=q_matrix,

    Iq_matrix=Iq_matrix,

    Isim_matrix=Isim_matrix,

    sigma_matrix=sigma_matrix,

    r_matrix=r_matrix,

    pr_matrix=pr_matrix,

    dmax_array=dmax_array,

    r_grid_matrix=r_grid_matrix,

    **combined_parameters,
)

# ============================================================
# DONE
# ============================================================

print()
print(
    f"Saved combined dataset to:\n"
    f"{OUTPUT_FILE}"
)

Loading: saxs_training_data_superellipsoid.npz
Loading: saxs_training_data_superellipsoid1.npz
Loading: saxs_training_data_superellipsoid2.npz
Loading: saxs_training_data_superellipsoid3.npz
Loading: saxs_training_data_superellipsoid4.npz

FINAL DATASET
-------------
q_matrix: (9000, 512)
Iq_matrix: (9000, 512)
Isim_matrix: (9000, 512)
sigma_matrix: (9000, 512)
r_matrix: (9000, 512)
pr_matrix: (9000, 512)
dmax_array: (9000,)
r_grid_matrix: (9000, 512)
shape_labels: (9000,)
model_labels: (9000,)

Saved combined dataset to:
C:\Users\Nikla\OneDrive\Skrivebord\BSc projekt\Code BSc Project\saxs_training_data_combined_superellipsoid.npz


In [14]:
from pathlib import Path
import numpy as np

DATASET_DIR = Path.cwd()

npz_files = [
    "saxs_training_data_superellipsoid.npz",
    "saxs_training_data_superellipsoid1.npz",
    "saxs_training_data_superellipsoid2.npz",
    "saxs_training_data_superellipsoid3.npz",
    "saxs_training_data_superellipsoid4.npz",
]


all_ok = True

reference_q = None
reference_r = None

for file_name in npz_files:

    path = DATASET_DIR / file_name

    print(f"\nChecking: {file_name}")

    data = np.load(path, allow_pickle=True)

    required = [
        "q_matrix",
        "Iq_matrix",
        "r_matrix",
        "pr_matrix",
        "dmax_array",
        "r_grid_matrix",
        "model_labels"
    ]

    # -------------------------
    # Required keys
    # -------------------------
    for key in required:
        if key not in data:
            print(f"  Missing key: {key}")
            all_ok = False

    # -------------------------
    # Shape checks
    # -------------------------
    n = data["Iq_matrix"].shape[0]

    for key in required:

        arr = data[key]

        if key == "dmax_array":

            if arr.shape != (n,):
                print(f"  Bad shape for {key}: {arr.shape}")

        else:

            if arr.shape != (n, 512):
                print(f"  Bad shape for {key}: {arr.shape}")

    # -------------------------
    # q-grid consistency
    # -------------------------
    q0 = data["q_matrix"][0]
    r0 = data["r_grid_matrix"][0]

    if reference_q is None:
        reference_q = q0
        reference_r = r0
    else:

        if not np.allclose(q0, reference_q):
            print("  WARNING: q-grid differs")

        if not np.allclose(r0, reference_r):
            print("  WARNING: r-grid differs")

print("\n====================")

if all_ok:
    print("Core dataset structure is VALID")
else:
    print("Dataset structure problems found")

print("====================")


Checking: saxs_training_data_superellipsoid.npz
  Bad shape for model_labels: (1000,)

Checking: saxs_training_data_superellipsoid1.npz
  Bad shape for model_labels: (2000,)

Checking: saxs_training_data_superellipsoid2.npz
  Bad shape for model_labels: (2000,)

Checking: saxs_training_data_superellipsoid3.npz
  Bad shape for model_labels: (2000,)

Checking: saxs_training_data_superellipsoid4.npz
  Bad shape for model_labels: (2000,)

Core dataset structure is VALID
